In [1]:
import cv2
import mediapipe as mp
import numpy as np
import math
import joblib
import time
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# ================= LOAD MODELS =================
air_model = joblib.load("svm_air_model.pkl")
air_scaler = joblib.load("scaler.pkl")

gesture_model = joblib.load("gesture_control_svm.pkl")
gesture_scaler = joblib.load("gesture_control_scaler.pkl")

GESTURE_CLASSES = gesture_model.classes_

# ================= CONFIG =================
HAND_MODEL_PATH = "hand_landmarker.task"
PINCH_THRESHOLD = 0.04
AIR_CONF_THRESHOLD = 0.75
RESAMPLE_POINTS = 100

# ================= HAND TRACKER =================
base_options = python.BaseOptions(model_asset_path=HAND_MODEL_PATH)

options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=1,
    running_mode=vision.RunningMode.IMAGE
)

detector = vision.HandLandmarker.create_from_options(options)

# ================= AIR PREPROCESS =================
def preprocess_stroke(points, num_points=100):
    if len(points) < 20:
        return None

    pts = np.array(points, dtype=np.float32)
    pts -= np.mean(pts, axis=0)
    pts /= (np.max(np.abs(pts)) + 1e-6)

    t_old = np.linspace(0, 1, len(pts))
    t_new = np.linspace(0, 1, num_points)

    x_interp = np.interp(t_new, t_old, pts[:, 0])
    y_interp = np.interp(t_new, t_old, pts[:, 1])

    return np.stack((x_interp, y_interp), axis=1).flatten()

# ================= MAIN =================
cap = cv2.VideoCapture(0)

points = []
output_text = ""
mode = "AIR"  # AIR or GESTURE

print("Controls:")
print("F - Switch Mode")
print("E - Confirm Letter (Air Mode)")
print("C - Clear Text")
print("Q - Quit")

while True:
    start_time = time.time()
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

    result = detector.detect(mp_image)

    predicted_gesture = None
    pinch_active = False

    if result.hand_landmarks:
        for hand_landmarks in result.hand_landmarks:

            h, w, _ = frame.shape

            # ================= PINCH =================
            thumb = hand_landmarks[4]
            index = hand_landmarks[8]

            dist = math.sqrt(
                (thumb.x - index.x)**2 +
                (thumb.y - index.y)**2
            )

            if dist < PINCH_THRESHOLD:
                pinch_active = True

            # ================= LANDMARKS =================
            landmarks = []
            for lm in hand_landmarks:
                landmarks.append([lm.x, lm.y])

            landmarks = np.array(landmarks, dtype=np.float32)

            # Flip left hand
            if result.handedness:
                hand_label = result.handedness[0][0].category_name
                if hand_label == "Left":
                    landmarks[:, 0] = 1.0 - landmarks[:, 0]

            # Gesture prediction
            features = landmarks.flatten().reshape(1, -1)
            features = gesture_scaler.transform(features)

            scores = gesture_model.decision_function(features)
            best_index = np.argmax(scores)
            predicted_gesture = GESTURE_CLASSES[best_index]

            # Draw landmarks
            for lm in hand_landmarks:
                cx, cy = int(lm.x * w), int(lm.y * h)
                cv2.circle(frame, (cx, cy), 3, (0, 255, 0), -1)

    # ================= AIR MODE =================
    if mode == "AIR":
        if pinch_active and result.hand_landmarks:
            hand_landmarks = result.hand_landmarks[0]
            h, w, _ = frame.shape
            index = hand_landmarks[8]
            ix, iy = int(index.x * w), int(index.y * h)
            points.append((ix, iy))

        for i in range(1, len(points)):
            cv2.line(frame, points[i-1], points[i], (255, 0, 0), 4)

        mode_text = "MODE: AIR WRITING"

    # ================= GESTURE MODE =================
    else:
        mode_text = "MODE: GESTURE RECOGNITION"

        if predicted_gesture:
            cv2.putText(frame,
                        f"Gesture: {predicted_gesture}",
                        (20, 120),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.9,
                        (0, 255, 0),
                        2)

    # ================= UI =================
    cv2.rectangle(frame, (0, 0), (frame.shape[1], 90), (40, 40, 40), -1)

    cv2.putText(frame, mode_text,
                (20, 35),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 255),
                2)

    latency = (time.time() - start_time) * 1000
    cv2.putText(frame, f"Latency: {latency:.1f} ms",
                (20, 70),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 200, 200),
                2)

    # Output at bottom
    cv2.putText(frame, output_text,
                (50, frame.shape[0] - 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                2,
                (0, 0, 255),
                3)

    cv2.imshow("Hybrid Air Writing + Gesture System", frame)

    key = cv2.waitKey(1) & 0xFF

    # ================= KEY CONTROLS =================
    if key == ord('f'):
        mode = "GESTURE" if mode == "AIR" else "AIR"
        points = []

    elif key == ord('e') and mode == "AIR":
        features = preprocess_stroke(points, RESAMPLE_POINTS)
        if features is not None:
            features = air_scaler.transform([features])
            probs = air_model.predict_proba(features)[0]
            if np.max(probs) > AIR_CONF_THRESHOLD:
                letter = air_model.classes_[np.argmax(probs)]
                output_text += letter
        points = []

    elif key == ord('c'):
        output_text = ""
        points = []

    elif key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

c:\Users\gayat\anaconda3\Lib\site-packages\scipy\__init__.py:143: UserWarning: A NumPy version >=1.19.5 and <1.27.0 is required for this version of SciPy (detected version 2.4.2)
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [1]:
conda create -n myenv python=3.10 numpy=1.26.4 scipy=1.11.4 scikit-learn joblib mediapipe opencv -y
conda activate myenv

SyntaxError: invalid syntax (101456503.py, line 1)

In [4]:
%pip install "numpy<2.0"

  Obtaining dependency information for numpy<2.0 from https://files.pythonhosted.org/packages/3f/6b/5610004206cf7f8e7ad91c5a85a8c71b2f2f8051a0c0c4d5916b76d6cbb2/numpy-1.26.4-cp311-cp311-win_amd64.whl.metadata
  Using cached numpy-1.26.4-cp311-cp311-win_amd64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp311-cp311-win_amd64.whl (15.8 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.2
    Uninstalling numpy-2.4.2:
      Successfully uninstalled numpy-2.4.2
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\gayat\\anaconda3\\Lib\\site-packages\\~umpy.libs\\libscipy_openblas64_-74a408729250596b0973e69fdd954eea.dll'
Consider using the `--user` option or check the permissions.



In [6]:
%pip install --user --force-reinstall "numpy<2.0"

  Obtaining dependency information for numpy<2.0 from https://files.pythonhosted.org/packages/3f/6b/5610004206cf7f8e7ad91c5a85a8c71b2f2f8051a0c0c4d5916b76d6cbb2/numpy-1.26.4-cp311-cp311-win_amd64.whl.metadata
  Using cached numpy-1.26.4-cp311-cp311-win_amd64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp311-cp311-win_amd64.whl (15.8 MB)
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.3.0 requires FuzzyTM>=0.4.0, which is not installed.
tables 3.8.0 requires blosc2~=2.0.0, which is not installed.
tables 3.8.0 requires cython>=0.29.21, which is not installed.
transformers 2.1.1 requires sentencepiece, which is not installed.
numba 0.57.0 requires numpy<1.25,>=1.21, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.


In [ ]:
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

{'status': 'ok', 'restart': True}

: 

In [ ]:
!conda install numpy=1.26.4 scipy=1.11.4 -y

In [ ]:
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

{'status': 'ok', 'restart': True}

: 